# Clusterização dos pagantes — Semana 5 (objetivo 3)

**Pergunta:** em quais segmentos os 14.753 pagantes ativos em 30/09/2026 se dividem, pelo comportamento, e como cada um se descreve?

**Regras:** churn, plano, canal, receita e renovação **não entram** no agrupamento: só descrevem os grupos depois. Nada posterior ao corte. Dados por assinante ficam em `dados/`, fora do Git.

**Entrada:** `dados/processados/s5_john/base_clusterizacao.csv` e `base_assinantes_corte.csv`, gerados por `src/preparar_diagnostico_s5.py` a partir do bruto (limpeza de fusos, contas de teste, durações zero, cópias app × relógio). O código desta etapa está em `src/clusterizacao_s5.py`.

In [ ]:
from pathlib import Path
import runpy, json
import numpy as np, pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'dados/processados/s5_john/base_clusterizacao.csv').exists(), 'rode antes src/preparar_diagnostico_s5.py'
cl = runpy.run_path(str(ROOT / 'src/clusterizacao_s5.py'))
base, contexto = cl['carregar']()
base.shape

## 1. Escolha das variáveis: uma por dimensão

As seis medidas de volume (treinos, minutos e dias ativos, em 30 e 90 dias) têm correlação de Spearman entre 0,80 e 0,99. Usá-las juntas daria a "quanto treina" um peso seis vezes maior que às outras dimensões, e os clusters virariam faixas de volume. Ficamos com uma variável por dimensão.

In [ ]:
candidatas = base.drop(columns=['user_id', 'nivel'])
rho = candidatas.corr(method='spearman')
pares = [(i, j, round(rho.loc[i, j], 2)) for i in rho for j in rho if i < j and abs(rho.loc[i, j]) >= .7]
pd.DataFrame(pares, columns=['var_1', 'var_2', 'spearman']).sort_values('spearman', ascending=False)

`tickets_90d` fica fora: 96% das pessoas têm zero chamados. Padronizada, quem abriu um ticket fica a cerca de 5 desvios-padrão da média, e o K-means criaria um grupo só por causa da escala. A variável entra apenas na descrição.

`nivel` e `dias_planejados_semana` são declarações do cadastro, não comportamento, e têm 7–10% de ausência: também só descrevem.

In [ ]:
(base.tickets_90d.value_counts(normalize=True) * 100).round(1)

## 2. Exposição: contas recentes não tiveram 90 dias

Quem criou a conta há 30 dias não teve como acumular 90 dias de treino, então pareceria pouco ativo por construção. Como registrar treino é grátis (o Premium começa com 7 dias de teste), o denominador é o tempo de **cadastro**, não o de assinatura: 206 pessoas têm mais dias ativos do que dias de assinatura, e nenhuma tem mais do que dias de conta + 1.

- frequência = dias ativos ÷ min(90, cadastro + 1); posts idem, reescalados para 90 dias;
- tendência (30 dias × 30 anteriores): com menos de 60 dias de conta, a janela anterior está incompleta e a variação sai positiva por construção. Fica neutra (0).

In [ ]:
exp = (contexto.set_index('user_id').tempo_cadastro_dias + 1).clip(upper=90)
b = base.set_index('user_id')
pd.Series({
    'dias ativos > dias de assinatura': int((b.dias_ativos_90d > b.tempo_assinatura_dias.clip(upper=90)).sum()),
    'dias ativos > cadastro + 1': int((b.dias_ativos_90d > exp).sum()),
    'exposição < 90 dias': int((exp < 90).sum()),
    'conta com menos de 60 dias': int((exp < 60).sum()),
})

## 3. Ausências e escala

As ausências são estruturais: os 177 sem recência são exatamente os 177 sem treino em 2026; os 759 sem proporção de vídeo são exatamente os que têm zero treino em 90 dias. A tabela abaixo registra cada regra e quantas linhas ela afetou. Contagens com cauda longa recebem `log1p`; todas são cortadas nos percentis 1 e 99 e padronizadas.

In [ ]:
f, x, decisoes = cl['preparar'](base, contexto)
display(pd.DataFrame(decisoes))
x.describe().T.round(2)

In [ ]:
rho_final = x.corr(method='spearman')
print('maior |rho| entre as variáveis finais:', rho_final.where(~np.eye(len(rho_final), dtype=bool)).abs().max().max().round(2))
rho_final.round(2)

## 4. Quantos grupos

Para cada k: inércia (cotovelo), silhueta, Calinski-Harabasz e **estabilidade**. A estabilidade ajusta o K-means em 20 amostras de 80% da base e mede, pelo ARI, se as pessoas caem nos mesmos grupos que no ajuste completo (1 = idêntico; 0 = acaso).

**Leitura honesta:** a silhueta fica perto de 0,17. Os assinantes não formam ilhas naturais; o comportamento é contínuo, e os segmentos são cortes estáveis desse contínuo. Por isso a estabilidade e a interpretação pesam mais que a silhueta na escolha.

In [ ]:
z = cl['padronizar'](x)
varredura = cl['varrer_k'](z)
varredura.round(3)

**Escolha: k = 4.** É o mais estável (ARI mínimo 0,98), o menor grupo tem 19% da base, separa os engajados entre vídeo em casa e ar livre (o que muda a mensagem no objetivo 6) e concorda mais com o hierárquico de Ward do que o k = 3. Com k = 5 a estabilidade cai (ARI mínimo 0,63).

In [ ]:
K = 4
rotulos = cl['ajustar'](z, K)
ordem = pd.Series(f.taxa_dias_ativos_90d.values).groupby(rotulos).mean().sort_values().index
rotulos = pd.Series(rotulos).map({old: new for new, old in enumerate(ordem)}).to_numpy()
print('ARI K-means × Ward (amostra de 6.000):', round(cl['comparar_ward'](z, rotulos, K), 3))
pd.DataFrame(z, columns=x.columns).groupby(rotulos).mean().round(2)

## 5. Tabela comparativa

Variáveis na escala original. Plano, canal, renovação e receita **não entraram** no agrupamento: aparecem só para descrever. A renovação desligada funciona como validação externa: um agrupamento feito apenas com comportamento de uso separou o grupo que já sinaliza saída.

In [ ]:
tabela, detalhado = cl['perfil'](f, contexto, rotulos)
assert tabela.assinantes.sum() == len(base) == 14753
tabela.round(2).T

## 6. Conferências

**Ponte de linhas:** nenhuma linha é removida; 14.753 entram e 14.753 saem com segmento.

**Conferência manual:** quatro assinantes recalculados direto de `dados/brutos/` com lógica independente (fuso do garmin, durações zero, cópias app × relógio): três de segmentos diferentes com relógio conectado e um de conta recente, para exercitar a exposição.

In [ ]:
a = pd.read_parquet(ROOT / 'dados/brutos/atividades.parquet')
u = pd.read_csv(ROOT / 'dados/brutos/usuarios.csv').set_index('user_id')
CUT = pd.Timestamp('2026-10-01')

def recalcular(uid):
    t = a[a.user_id == uid].copy()
    t['ts'] = pd.to_datetime(t.inicio)
    t.loc[t.origem == 'garmin', 'ts'] -= pd.Timedelta(hours=3)
    t = t[(t.ts >= '2026-01-01') & (t.ts < CUT) & (t.duracao_min > 0)]
    m = t[t.origem == 'app'].merge(t[t.origem != 'app'], on=['tipo', 'duracao_min'], suffixes=('_a', '_r'))
    t = t[~t.atividade_id.isin(m.loc[(m.ts_a - m.ts_r).abs() <= pd.Timedelta(minutes=5), 'atividade_id_a'])]
    j = t[t.ts >= CUT - pd.Timedelta(days=90)]
    exp = min(90, (pd.Timestamp('2026-09-30') - pd.to_datetime(u.loc[uid, 'data_cadastro']).normalize()).days + 1)
    return {'dias_ativos_90d': j.ts.dt.date.nunique(), 'exposicao_dias': exp,
            'proporcao_video_90d': round(j.tipo.isin(['forca', 'strength', 'funcional', 'hiit', 'yoga']).mean(), 3),
            'recencia_dias': round((CUT - t.ts.max()).total_seconds() / 86400, 2),
            'min_por_treino_90d': round(j.duracao_min.mean(), 3)}

conf = []
for uid in ['u_a40c9770', 'u_6fe212e4', 'u_e21acf36', 'u_f9d0da76']:
    p = detalhado.set_index('user_id').loc[uid]
    for k, v in recalcular(uid).items():
        conf.append({'user_id': uid, 'segmento': p.segmento, 'metrica': k, 'manual': v, 'pipeline': round(p[k], 3)})
conf = pd.DataFrame(conf)
conf['bate'] = np.isclose(conf.manual.astype(float), conf.pipeline.astype(float), atol=0.01)
assert conf.bate.all()
conf